### Dataset and Task Metadata

In [97]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="emscad",
    dataset_year="2014",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction/data",
    download_description="""
The original website (http://icsdweb.aegean.gr/emscad) has been down for a while.
Thus, we us an alternative source from Kaggle (https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction/data).
There is also an additional alternative source from Kaggle (https://www.kaggle.com/datasets/amruthjithrajvr/recruitment-scam).

kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction -f fake_job_postings.csv && unzip fake_job_postings.csv.zip fake_job_postings.csv&& rm fake_job_postings.csv.zip
mkdir -p local-data-warehouse/emscad && mv fake_job_postings.csv local-data-warehouse/emscad
""",
    # References
    academic_reference_bibtex="""@article{vidros2017automatic,
  title={Automatic detection of online recruitment frauds: Characteristics, methods, and a public dataset},
  author={Vidros, Sokratis and Kolias, Constantinos and Kambourakis, Georgios and Akoglu, Leman},
  journal={Future Internet},
  volume={9},
  number={1},
  pages={6},
  year={2017},
  publisher={MDPI}
}
""",
    academic_reference_bibtex_key="vidros2017automatic",
    licence="CC0: Public Domain", # from Kaggle, paper does not state anything, website is down to check -> so maybe None...
    data_tags=["IID", "Spatial"],
    curation_comments="""
We have no access to the original state and the data already contains some preprocessed features.
The paper describes them in more detail as well in Table 2.

In the original paper, the authors subsampled the data to have 450 fraudulent and 450 non-fraudulent samples.
No details are given on how the subsampling was done. Moreover, duplicates were skipped, but no details are given on how duplicates were identified.

- The dataset was collected from 2012 to 2014. But the data contains no feature to identify the time of a sample. So we can expect that some bias from temporal leakage. Moreover, the data might contain multiple job postings from the same fraudster, which would need to be grouped, but we cannot identify them from the data.
- Following the original paper, we drop all duplicates (when ignoring the job_id column) to avoid data leakage.
- The data has a location description, we do not resolve it lat/longitude but leave it to the pipelines.
- The salary range has several weird quirks. It contains data either in the thousands, or is missing the "k" to denote thousands. Moreover, it contains empty or 0-0 entries. The column might contain yearly salary, hourly salary, or one-time payment. Finally, for some jobs, the column contains a date (e.g. "Oct-20"). We keep the column as complex as it is. We add one column that contains the maximum salary parsed from the text and we only keep salary values above 50k as valid values to create a numerical feature for differences in the larger ranges.
- We removed job listings with non english texts (138 rows).
- We do not subsample the data, as a result, the data is heavily imbalanced.
- We found no relation of job_id with a timestamp and even found cases where a lower job_id has a description claiming to be a job from 2014 (so the end of the collection period). Thus, we do dropped job_id as it does not contain any signal.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="fraudulent",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="fraudulent",
)

## Preprocessing

In [223]:
import pandas as pd
import  numpy as np
df = pd.read_csv(dataset_mold.path / "fake_job_postings.csv")
print("Loaded data shape:", df.shape)

# Drop duplicates
df = df[~df.drop(columns=["job_id"]).duplicated(keep="last")]

# Get max salary proxy for larger ranges
s = df["salary_range"].astype(str).str.strip()
invalid_mask = (df["salary_range"].isna() | (s == '') | (s == '0-0'))
ranges = s.str.split('-', expand=True)
min_vals, max_vals = ranges[0], ranges[1]
is_not_salary_range = pd.to_numeric(max_vals, errors="coerce").isna() | pd.to_numeric(min_vals, errors="coerce").isna() # remove dates
not_above_50_k =  ~(pd.to_numeric(max_vals, errors="coerce") >= 50_000)
df["salary_range_max"] = pd.to_numeric(max_vals, errors="coerce")
df.loc[invalid_mask | is_not_salary_range |not_above_50_k, "salary_range_max"] = np.nan

# from langdetect import detect, LangDetectException
#
# def is_not_english(text: str) -> bool:
#     try:
#         return detect(text) != "en"
#     except LangDetectException:
#         # Raised for very short or ambiguous text
#         return False
# df["is_english_description"] = df["description"].astype(str).apply(lambda x: not is_not_english(x))
# bad_filter_list = [
#     # Missing, faulty or, Lorem ipsum description...
#     1243,
#     3031,
#     7613,
#     5557,
#     11078,
#     11894,
#     12190,
#     13530,
# ]
# print(list(df[(~df["is_english_description"]) & (~df["description"].isna()) & (df["description"] != "Sales Executive") & (~df["job_id"].isin(bad_filter_list))]["job_id"]))
# Output from the above
filter_non_en_job = [543, 550, 557, 687, 936, 986, 1174, 1558, 1667, 1689, 1794, 1831, 1922, 2351, 2384, 2424, 2474, 2677, 2788, 2857, 2929, 2967, 2986, 3126, 3132, 3139, 3156, 3169, 3566, 3576, 3702, 3764, 3856, 3952, 3993, 4157, 4185, 4193, 4315, 4362, 4371, 4401, 4812, 4825, 4867, 5058, 5198, 5274, 5632, 5637, 5651, 5693, 5724, 5931, 6023, 6096, 6169, 6184, 6269, 6284, 6438, 6445, 6573, 6587, 6699, 6752, 6853, 6988, 7025, 7112, 7212, 7549, 7679, 7849, 7881, 8066, 8106, 8214, 8355, 8365, 8397, 8615, 8719, 8791, 8895, 9216, 9393, 9413, 9965, 10450, 10468, 10557, 10570, 10661, 11075, 11314, 11320, 11383, 11419, 11424, 11426, 11427, 11446, 11449, 11463, 11464, 11536, 11631, 11748, 11958, 12231, 12395, 12683, 12973, 13016, 13301, 13474, 13476, 13551, 13696, 14021, 14131, 14333, 14911, 15185, 15410, 16343, 16500, 16586, 16608, 16691, 17034, 17047, 17148, 17226, 17328, 17357, 17781]
df = df[~df["job_id"].isin(filter_non_en_job)]

as_string_col = [
    "title",
    "location",
    "department",
    "salary_range",
    # HTML fragments
    "company_profile",
    "description",
    "requirements",
    "benefits",
]
as_cat_type = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "fraudulent",
]

for c in as_string_col:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["job_id"]).reset_index(drop=True)

# After our preprocessing, there is just one entry without a description, so we drop it as well.
df = df[~df["description"].isna()].reset_index(drop=True)

## Data Checks

In [224]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 17,460
Columns: 18

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [225]:
# Sample Rows
df_head

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,salary_range_max
0,Marketing Intern,"US, NY, New York",Marketing,<NA>,"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City.","Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is currently interviewing full- and part-time unpaid interns to work in a small team of editors, executives, and developers in its New York City headquarters.Reproducing and/or repackaging existing Food52 content for a number of partner sites, such as Huffington Post, Yahoo, Buzzfeed, and more in their various content management systemsResearching blogs and websites for the Provisions by Food52 Affiliate ProgramAssisting in day-to-day affiliate program support, such as screening affiliates and assisting in any affiliate inquiriesSupporting with PR &amp; Events when neededHelping with office administrative work, such as filing, mailing, and preparing for meetingsWorking with developers to document bugs and suggest improvements to the siteSupporting the marketing and executive staff","Experience with content management systems a major plus (any blogging counts!)Familiar with the Food52 editorial voice and aestheticLoves food, appreciates the importance of home cooking and cooking with the seasonsMeticulous editor, perfectionist, obsessive attention to detail, maddened by typos and broken links, delighted by finding and fixing themCheerful under pressureExcellent communication skillsA+ multi-tasker and juggler of responsibilities big and smallInterested in and engaged with social media like Twitter, Facebook, and PinterestLoves problem-solving and collaborating to drive Food52 forwardThinks big picture but pitches in on the nitty gritty of running a small company (dishes, shopping, administrative support)Comfortable with the realities of working for a startup: being on call on evenings and weekends, and working long hours",<NA>,0,1,0,Other,Internship,NaN,NaN,Marketing,0,NaN
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,<NA>,"90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the worlds Cloud Video Production Service enabling brands and agencies to get high quality online video content shot and produced anywhere in the world. 90 Seconds makes video production fast, affordable, and all managed seamlessly in the cloud from purchase to publish. http://90#URL_fbe6559afac620a3cd2c22281f7b8d0eef56a73e3d9a311e2f1ca13d081dd630#90 Seconds removes the hassle, cost, risk and speed issues of working with regular video production companies by managing every aspect of video projects in a beautiful online experience. With a growing global network of over 2,000 rated video professionals in over 50 countries managed by dedicated production success teams in 5 countries, 90 Seconds provides a 100% success guarantee.90 Seconds has produced almost 4,000 videos in over 30 Countries for over 500 Global brands including some of the worlds largest i

In [226]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,required_education,category,7932,45.43,13,"Bachelor's Degree, High School or equivalent, Unspecified, Master's Degree, Associate Degree, Certification, Some College Coursework Completed, Professional, Vocational, Some High School Coursework"
1,required_experience,category,6890,39.46,7,"Mid-Senior level, Entry level, Associate, Not Applicable, Director, Internship, Executive"
2,function,category,6310,36.14,37,"Information Technology, Sales, Engineering, Customer Service, Marketing, Administrative, Design, Health Care Provider, Education, Other"
3,industry,category,4778,27.37,131,"Information Technology and Services, Computer Software, Internet, Education Management, Marketing and Advertising, Financial Services, Hospital & Health Care, Consumer Services, Telecommunications, Oil & Energy"
4,employment_type,category,3376,19.34,5,"Full-time, Contract, Part-time, Temporary, Other"
5,telecommuting,category,0,0.00,2,"0, 1"
6,has_company_logo,category,0,0.00,2,"1, 0"
7,has_questions,category,0,0.00,2,"0, 1"
8,fraudulent,category,0,0.00,2,"0, 1"
9,salary_range_max,float64,15972,91.48,119,"50000.0, 100000.0, 80000.0, 60000.0, 70000.0, 120000.0, 90000.0, 75000.0, 65000.0, 150000.0"


In [227]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
salary_range_max,1488.0,1.530745e+06,3.737839e+07,50000.0,1.200000e+09


In [228]:
# Categorical Feature Statistics
cat_stats

value  \
column              rank                                                                        
benefits            1                                                                    <NA>   
                    2                                                     See job description   
                    3                                                       Career prospects.   
                    4     CSD offers a competitive benefits package for full-time employee...   
                    5     Plenty of perksAs well as the opportunity to solve complex probl...   
company_profile     1                                                                    <NA>   
                    2                   We help teachers get safe &amp; secure jobs abroad :)   
                    3     We Provide Full Time Permanent Positions for many medium to larg...   
                    4     Novitex Enterprise Solutions, formerly Pitney Bowes Management S...   
                    5     Established on the principles that full time education is not fo...   
department          1                                                                    <NA>   
                    2                                                                   Sales   
                    3                                                             Engineering   
                    4                                                               Marketing   
                    5                                                              Operations   
description         1     Play with kids, get paid for it Love travel? Jobs in Asia$1,500+...   
                    2     Play with kids, get paid for it :-)Love travel? Jobs in Asia$150...   
                    3     Play with kids, get paid for it Love travel? Jobs in Asia$1500 U...   
                    4     Play with kids, get paid for it :-)Love travel? Jobs in Asia$150...   
                    5     The International Broadcaster shall properly complete all daily ...   
employment_type     1                                                               Full-time   
                    2                                                                    <NA>   
                    3                                                                Contract   
                    4                                                               Part-time   
                    5                                                               Temporary   
fraudulent          1                                                                       0   
                    2                                                                       1   
function            1                                                                    <NA>   
                    2                                                  Information Technology   
                    3                                                                   Sales   
                    4                                                             Engineering   
                    5                                                        Customer Service   
has_company_logo    1                                                                       1   
                    2                                                                       0   
has_questions       1                                                                       0   
                    2                                                                       1   
industry            1                                                                    <NA>   
                    2                                     Information Technology and Services   
                    3                                                       Computer Software   
                    4                                                                Internet   
              

In [229]:
# Target Distribution
target_df

,count,pct
fraudulent,,
0,16607,95.11
1,853,4.89


## Task Curation

In [230]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import RepeatedStratifiedKFold

n_repeats, n_splits = 3, 3
assert task_mold.stratify_on == task_mold.target_column_name
sklearn_splits = RepeatedStratifiedKFold(
    n_repeats=n_repeats, n_splits=n_splits, random_state=42
).split(X=df.drop(columns=[task_mold.target_column_name]), y=df[task_mold.target_column_name])

splits = {}
for split_i, (train_idx, test_idx) in enumerate(sklearn_splits):
    repeat_i = split_i // n_splits
    fold_i = split_i % n_splits
    if repeat_i not in splits:
        splits[repeat_i] = {}
    splits[repeat_i][fold_i] = (train_idx.tolist(), test_idx.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="In the original paper, 10-fold CV is used. We follow our default suggestions in the case of IID.",
    splits=splits,
)

## Export

In [231]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019bf56c-467d-7805-baab-94b17ee9dc04
aeb4f16183e3e5201ba5042e0d4db89bd491e06cf0a63e15b0128722af75ee9c
